<a href="https://colab.research.google.com/github/tanishi-mechatoro/rq.singyura/blob/main/notebooks/guide_and_experiments_ja.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RoboQuest 2026 — 解説・実験編
[クイックスタート編](https://colab.research.google.com/github/SingularityBattleQuest/RoboQuest2026/blob/main/notebooks/quickstart_ja.ipynb) と同じモデルを使い、仕組みを理解しながらコードを変更します。初めてでも上から実行できます。

**目標：観測・行動・報酬の関係を説明し、設定を1つ変えて結果を比較すること。**
学習や保存の処理は両教材で共通です。クイックスタートの実験フォルダを指定すれば、再学習せずにモデルを表示・評価できます。別の実験を行うときは実験名を変えてください。


ローカルで検証した環境を再現する場合は、配布された `walk_colab_bundle.zip` をColab左側の「ファイル」にアップロードしてからセットアップを実行します。ライブラリの版と学習コードを揃えますが、MacとColabで計算結果が完全に同一になることを保証するものではありません。


新規学習は、見本の学習と約1万ステップのPPO微調整を行います。保存済みモデルを使う場合はこの学習時間を省けます。

現在の初期設定は、約1.5 Hzのゆっくりした脚運び・停止・再発進の見本を先に覚え、PPOで微調整する方式です。教師あり学習と強化学習を組み合わせています。停止中にモデル自身が取る姿勢からの再発進例も追加学習します。高さを保ちながら、速すぎる関節運動と足先以外の床接触を減点します。

GitHubに同梱した検証済みモデル（ZIPをアップロードした場合はZIP内のモデル）は `retrain_walk=False` で読み込めます。初期設定は `True` で、同梱モデルを使わず未学習状態から学習します。前進・停止・再発進の実測評価に合格するまで先へ進みません。歩行学習の初期値は見本約3万ステップ＋PPO約1万ステップで、PPOの入力正規化を固定しています。学習率や追加ステップ数を大きくすると、覚えた歩き方が崩れる場合があります。後退・横移動・旋回は別に評価が必要です。

保存済み歩行モデルを見る場合は、セットアップ後に末尾の「保存したモデルを使う」へ進んでください。


In [3]:
#@title 🔧 セットアップ（最初に一度だけ実行してください）

import subprocess, sys, os, zipfile
SETUP_READY = False

print(f'Python {sys.version_info.major}.{sys.version_info.minor} で実行中')

subprocess.run(
    'command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -q ffmpeg)',
    shell=True, check=False)

# ローカル検証済みのコード一式を使う場合は、先にこのZIPをColabにアップロード。
_bundle = '/content/walk_colab_bundle.zip'
if os.path.isfile(_bundle):
    with zipfile.ZipFile(_bundle) as archive:
        for member in archive.namelist():
            destination = os.path.realpath(os.path.join('/content', member))
            if not destination.startswith('/content/RoboQuest2026/'):
                raise ValueError('想定外のファイル名を含むZIPです。')
        archive.extractall('/content')
    print('ローカルと同じコード・モデルを読み込みました。')
elif not os.path.exists('/content/RoboQuest2026/.git'):
    print('リポジトリをダウンロード中...')
    subprocess.run(['git', 'clone', '-q',
        'https://github.com/SingularityBattleQuest/RoboQuest2026.git',
        '/content/RoboQuest2026'], check=True)
else:
    print('リポジトリを最新化中...')
    subprocess.run(['git', '-C', '/content/RoboQuest2026', 'pull', '--ff-only', 'origin', 'main', '-q'],
                   check=False)

os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

# Colab本体のライブラリは変更せず、学習専用の環境を作ります。
print('学習用環境を準備中（初回は数分かかります）...', flush=True)
from scripts.colab_runtime import PYTHON as TRAINING_PYTHON, prepare_runtime
prepare_runtime()
MJSWAN_AVAILABLE = True
SETUP_READY = True
print('✅ セットアップ完了！再起動せず、次のセルへ進んでください。')


Python 3.13 で実行中
リポジトリをダウンロード中...
学習用環境を準備中（初回は数分かかります）...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 53.4 MB/s eta 0:00:00
PyTorch公式CPU版 2.14.0+cpu をインストールします。
Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.3/196.3 MB 24.8 MB/s  0:00:07
CPU学習用PyTorch: 2.14.0+cpu
=== Go2 MuJoCo モデルのダウンロード ===
保存先: /content/RoboQuest2026/models/go2

[1/2] ルートファイルを取得...
  スキップ (既存): go2.xml
  スキップ (既存): scene.xml
  スキップ (既存): scene_terrain.xml

[2/2] アセット(.obj)を取得...
  スキップ (既存): base_0.obj
  スキップ (既存): base_1.obj
  スキップ (既存): base_2.obj
  スキップ (既存): base_3.obj
  スキップ (既存): base_4.obj
  スキップ (既存): calf_0.obj
  スキップ (既存): calf_1.obj
  スキップ (既存): calf_mirror_0.obj
  スキップ (既存): calf_mirror_1.obj
  スキップ (既存): foot.obj
  スキップ (既存): hip_0.obj
  スキップ (既存): hip_1.obj
  スキップ (既存): thigh_0.obj
  スキップ (既存): thigh_1.obj
  スキップ (既存): thigh_mirror_0.obj
  スキップ (既存): thigh_mirror_1.obj

✅ 完了: go2.xml が正常に取得されました
学習用環境のモデル読み込み確認: OK
✅ セットアップ完了！再起動せず、次

In [4]:
#@title 📁 保存先（実験ごとに experiment_name を変える）
team_name = '隠居したいンゴ' #@param {type:"string"}
experiment_name = 'shiken_4' #@param {type:"string"}
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
for value in (team_name, experiment_name):
    if not value or value in ('.', '..') or '/' in value or '\\' in value:
        raise ValueError('チーム名・実験名にはフォルダ名を1つ指定してください')
SAVE_DIR = Path('/content/drive/MyDrive/RoboQuest2026') / team_name / experiment_name
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'保存先: {SAVE_DIR}')


Mounted at /content/drive
保存先: /content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4


In [5]:
#@title 共通機能を読み込む
#@markdown 設定を直接編集するときは、セルのメニューからコードを表示してください。
# 共通機能（学習処理は scripts/notebook_workflow.py）
if not globals().get('SETUP_READY', False):
    raise RuntimeError('先にセットアップを完了してください。')
from pathlib import Path
from copy import deepcopy
from scripts.colab_client import notebook_api
globals().update(notebook_api())


## 1. ロボットは何を学ぶ？
強化学習では、ロボットが **観測 → 行動 → 報酬** を繰り返し、累積報酬を大きくする方策（ポリシー）を学びます。ここではPPOという手法と、数値ベクトルを処理するMLPを使います。

2つの役割を分ける階層型です。

**鬼の位置や姿勢 → 逃げモデル → 速度指令 → 歩行モデル → 12関節の目標角度 → シミュレーター**

歩行モデルを先に学習し、逃げ学習中は固定します。逃げモデルは足の動かし方を一から覚えず、移動方向の選択に集中できます。本番の新仕様は高レベル19次元、低レベル47次元です。旧モデルは互換性対象外です。

### 歩行モデル：観測45次元 → 行動12次元
| 観測の範囲 | 内容 |
|---|---|
| `[0:3]` | 前後・左右・旋回の速度指令 |
| `[3:6]` | 胴体の角速度 |
| `[6:9]` | 胴体から見た重力方向 |
| `[9:21]` | 立ち姿勢からの関節角度差 |
| `[21:33]` | 関節角速度 |
| `[33:45]` | 前回の行動 |

行動は `[-1, 1]` の12個の値です。`立ち姿勢 + 0.6 × 行動` を目標関節角度に変換します。制御周期は50Hzです。


## 2. 歩行の報酬を設計する
速度追跡の報酬は指令への近さを評価します。重みを増やすことは、速度そのものを指定することとは違います。
姿勢の傾き・トルク・行動の急変は、負の重みでペナルティにします。例えば `orientation_weight` を `-1.0` から `-2.0` にすると傾きを強く罰します。転倒ペナルティは正の大きさを設定し、環境側で引きます。

他にも足のリズムや滑りの項があります。まずは1項目だけ変えて、同じ速度指令で歩行を比較しましょう。報酬を変えても望んだ動きになる保証はなく、他の項との釣り合いが大切です。


In [6]:
#@title 歩行報酬の設定
#@markdown 設定を直接編集するときは、セルのメニューからコードを表示してください。
walk_cfg = WalkRewardConfig(**WALK_FORWARD_REWARD)
# 例: walk_cfg.lin_vel_weight = 4.0
#@title 歩行報酬のカスタマイズ設計
# デフォルトの設定（WALK_FORWARD_REWARD）をベースに特定の報酬重みを変更します
custom_walk_cfg = WalkRewardConfig(**WALK_FORWARD_REWARD)

# --- 報酬設計のパラメータ変更（例） ---
# 1. 前進速度をより正確に追従させたい場合（初期値より増やす）
custom_walk_cfg.lin_vel_weight = 4.0

# 2. 胴体の傾きやブレをより厳しく抑制したい場合（マイナスペナルティを大きくする）
custom_walk_cfg.orientation_weight = -2.0

# 3. 関節の急激な動きや摩擦ロスを抑えるペナルティを調整（必要に応じてコメントアウトを解除）
# custom_walk_cfg.action_rate_weight = -0.02
# custom_walk_cfg.torques_weight = -0.0001

# 適用された設定を walk_cfg に格納し、現在の重みを出力して確認します
walk_cfg = custom_walk_cfg
print("=== カスタマイズした歩行報酬設定 ===")
for key, val in vars(walk_cfg).items():
    print(f"{key}: {val}")



=== カスタマイズした歩行報酬設定 ===
linear_tracking_variance: 0.16
angular_tracking_variance: 0.04
lin_vel_weight: 4.0
ang_vel_weight: 3.0
orientation_weight: -2.0
base_height_target: 0.29
base_height_weight: -500.0
vertical_velocity_weight: -2.0
nonfoot_contact_weight: -2.0
knee_height_target: 0.1
knee_height_weight: -1000.0
torques_weight: -2.5e-05
action_rate_weight: -0.05
joint_velocity_weight: -0.003
stand_joint_velocity_weight: -0.015
feet_gait_weight: 0.0
diagonal_support_weight: 0.0
foot_slip_weight: -0.02
fall_penalty: 10.0


### PPOの設定とネットワーク
| 設定 | 意味・変更時の考え方 |
|---|---|
| `learning_rate` | 1回の更新の大きさ。小さくすると更新が慎重になりますが、学習が遅くなる場合があります |
| `n_steps` | 各環境で更新前に集めるステップ数 |
| `batch_size` | 更新に使う小分けデータの数。`n_steps × num_envs` を割り切れる値が扱いやすいです |
| `n_epochs` | 集めたデータを繰り返し学習する回数 |
| `gamma` | 将来の報酬をどれだけ重視するか |
| `gae_lambda` | 価値推定におけるバイアスと分散のバランス |
| `ent_coef` | 行動の多様性を促す強さ |
| `net_arch` | 隠れ層のユニット数。大きくすると計算量も増えます |

`timesteps` は全環境で集める合計ステップ数で、更新単位によって指定値を超えることがあります。`num_envs` は並列の環境数ですが、この教材の実行方式では増やすだけで必ず高速化するわけではありません。
`seed` は乱数の初期値です。同じ条件で比較する助けになりますが、実行環境をまたぐ完全一致を保証しません。


In [7]:
#@title 歩行学習の設定
#@markdown 設定を直接編集するときは、セルのメニューからコードを表示してください。
seed = 0
walk_timesteps = WALK_FORWARD_STEPS
walk_num_envs = 4
walk_ppo = deepcopy(WALK_PPO)
walk_ppo['learning_rate'] = 1e-6
walk_ppo['ent_coef'] = 0.01
walk_ppo['policy_kwargs']['net_arch'] = [256, 256, 128]
print(walk_ppo)

{'learning_rate': 1e-06, 'n_steps': 1024, 'batch_size': 256, 'n_epochs': 5, 'gamma': 0.99, 'gae_lambda': 0.95, 'ent_coef': 0.01, 'target_kl': 0.001, 'policy_kwargs': {'net_arch': [256, 256, 128]}, 'device': 'cpu'}


In [8]:
#@title 🚀 歩行モデルを学習する（初期設定のまま実行・合格したモデルを保存）
#@markdown 初期設定では未学習のニューラルネットから、見本の学習＋PPOで前進・停止・再発進を覚えます。学習と評価に数分かかります。
#@markdown 学習済みモデルを試すだけの場合はチェックを外します（新規学習ではありません）。
retrain_walk = False #@param {type:"boolean"}
walk_export_ready = False
walk_ready = False
if retrain_walk:
    walk_report = learn_walk(SAVE_DIR, steps=walk_timesteps, seed=seed,
        reward_config=walk_cfg, ppo_kwargs=walk_ppo, num_envs=walk_num_envs)
else:
    bundle_dir = Path('models/teams/bundled_walk')
    if not bundle_dir.is_dir():
        bundle_dir = Path('models/pretrained/smooth_walk')
    load_bundled_walk(SAVE_DIR, bundle_dir)
    walk_report = qualify_walk(SAVE_DIR)
if not walk_report['passed']:
    raise RuntimeError('歩行評価に不合格です。walk_qualification.jsonを確認してください。')
walk_ready = True
print(f'✅ 前進・停止・再発進に合格したモデル: {SAVE_DIR}')


ローカルで保存したモデル・正規化データ・設定を読み込みました。
stand falls 0 velocity [ 0. -0. -0.]
forward falls 0 velocity [ 0.43  -0.001 -0.001]
停止・再発進: 12/12 区間合格
✅ 前進・停止・再発進に合格したモデル: /content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4


In [9]:
#@title 📊 歩行獲得を確認する（学習前後・前進・停止・再発進）
from scripts.learn_walk import require_qualified_walk
walk_report = require_qualified_walk(SAVE_DIR)
for label, report in [('学習前', walk_report.get('untrained')), ('学習後', walk_report['fixed'])]:
    if report is None:
        continue
    for command in ('stand', 'forward'):
        rows = [r for r in report['rows'] if r['command'] == command]
        speeds = [r['mean_velocity'][0] for r in rows if r['mean_velocity'][0] is not None]
        speed = sum(speeds) / len(speeds) if speeds else 0.0
        print(f"{label} {command}: 平均速度 {speed:.3f} m/s / 転倒 {sum(r['fell'] for r in rows)}/{len(rows)} / 合格 {sum(r['passed'] for r in rows)}/{len(rows)}")
print('✅ 20秒の前進・停止と、リセットなしの停止→前進→停止→再発進に合格。')
print('後退・横移動・旋回はこの入門モデルの合格範囲に含みません。')
# 必要な場合だけ、全方向の未達項目も確認します。
evaluate_all_directions = True #@param {type:"boolean"}
if evaluate_all_directions:
    all_directions_report = evaluate_walk(SAVE_DIR)
    print('全方向合格' if all_directions_report['passed'] else '全方向の課題は未達です。evaluation.jsonを確認してください。')


学習後 stand: 平均速度 0.000 m/s / 転倒 0/3 / 合格 3/3
学習後 forward: 平均速度 0.430 m/s / 転倒 0/3 / 合格 3/3
✅ 20秒の前進・停止と、リセットなしの停止→前進→停止→再発進に合格。
後退・横移動・旋回はこの入門モデルの合格範囲に含みません。
stand falls 0 velocity [ 0. -0. -0.]
forward falls 0 velocity [ 0.43  -0.001 -0.001]
backward falls 0 velocity [0. 0. 0.]
left falls 0 velocity [ 0. -0.  0.]
right falls 0 velocity [ 0. -0. -0.]
turn_left falls 0 velocity [0. 0. 0.]
turn_right falls 0 velocity [ 0. -0. -0.]
全方向の課題は未達です。evaluation.jsonを確認してください。


In [10]:
#@title 🎥 学習した歩行を動画で見る（前進12秒）
from scripts.learn_walk import require_qualified_walk
require_qualified_walk(SAVE_DIR)
walk_video = SAVE_DIR / 'walk_learned.mp4'
record_walk(SAVE_DIR, walk_video, seconds=12, command=(0.4, 0.0, 0.0),
            fps=10, width=480, height=320)
from IPython.display import Video, display
display(Video(str(walk_video), embed=True, html_attributes='controls autoplay muted loop'))


歩行動画を作成中（12秒、480×320）...
  動画 2/12 秒
  動画 4/12 秒
  動画 6/12 秒
  動画 8/12 秒
  動画 10/12 秒
/content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4/walk_learned.mp4


### 歩行を確認する
前後・左右・旋回の指令をそれぞれ試し、転倒しないか、指令に従うかを見ます。まず歩行を安定させてから逃げ学習に進んでください。


### 実行が止まったとき
- セットアップ完了後は再起動せず、次のセルへ進めます。初回は学習用ライブラリのダウンロードに数分かかります。途中で通信が切れた場合はセットアップを再実行してください。
- 学習・評価は専用環境のPyTorch公式CPU版で実行します。Colab本体のライブラリを置き換えないため、読み込み済みライブラリとの衝突を避けられます。CPUランタイムで学習できます。
- `retrain_walk=False` では同梱モデルをコピーするだけです。同梱ファイルがなければエラーで止まり、自動で新規学習は始めません。
- ビューアー用のONNX変換は、別セル・別プロセスで実行します。変換に失敗しても、保存済みのZIP・PKLから逃げ学習と数値評価を進められます。`walk_viewer_export.log` と終了コードを講師に共有してください。マシン全体のメモリ不足による停止まで防ぐものではありません。
- ランタイムが再起動した場合は「セットアップ」「保存先」「共通機能」を実行してから必要な評価・変換セルへ進みます。保存済みモデルがある場合、学習セルの再実行は不要です。


In [11]:
#@title 🌐 ビューアー用に変換（表示するときだけ実行・再学習不要）
#@markdown 学習・数値評価には不要です。失敗した場合はログを確認し、このセルだけ再実行してください。
import os, subprocess, sys
from pathlib import Path
from scripts.colab_runtime import worker_env

from scripts.learn_walk import require_qualified_walk
walk_export_ready = False
require_qualified_walk(SAVE_DIR)
export_log = SAVE_DIR / 'walk_viewer_export.log'
print(f'ビューアー用に変換中。ログ: {export_log}', flush=True)
with export_log.open('w', encoding='utf-8') as log:
    result = subprocess.run([
        str(TRAINING_PYTHON), '-X', 'faulthandler', '-u', 'scripts/export_for_web.py',
        '--only', 'walk', '--walk_model', str(SAVE_DIR / 'walk_model'),
        '--walk_vecnorm', str(SAVE_DIR / 'walk_model_vecnorm.pkl'),
        '--save_dir', '/content/RoboQuest2026/webapp/models',
    ], stdout=log, stderr=subprocess.STDOUT,
       env=worker_env())
print(export_log.read_text(encoding='utf-8')[-12000:])
if result.returncode != 0:
    raise RuntimeError(
        f'ビューアー用変換が失敗しました（終了コード {result.returncode}）。'
        f'ログ: {export_log}。保存済みモデルの再学習は不要です。'
        '学習・数値評価はそのまま進められます。ビューアーは変換成功後に実行してください。')
walk_export_ready = True
print('✅ ビューアー用変換が完了しました。')


ビューアー用に変換中。ログ: /content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4/walk_viewer_export.log
Loading PPO model: /content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4/walk_model
/content/RoboQuest2026/scripts/export_for_web.py:91: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(wrapper, dummy_obs, str(output_path), **export_kwargs)
Exported ONNX: /content/RoboQuest2026/webapp/models/walk_policy.onnx
Loading PPO model: /content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4/walk_model
  VecNormalize stats embedded from /content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4/walk_model_vecnorm.pkl
Exported ONNX: /content/RoboQuest2026/webapp/models

In [12]:
#@title 🎮 歩行ビューアー（mjswan — ブラウザ内 MuJoCo + 学習済みポリシー）
#@markdown 学習したモデルが前進指令0.4 m/sで動き始めます。Forwardを0にすると停止します。後退・横移動・旋回は未学習です。
#@markdown パネルのスライダー（Forward / Lateral / Yaw）で速度コマンドを入力してください。
#@markdown キー操作は `c`（パネルの開閉）と `r`（リセット）のみです。
#@markdown ※ 最初のビューアー実行時はブラウザ用の表示エンジンをビルドするため1〜3分ほどかかります（2回目以降はすぐ表示されます）。

if not globals().get('walk_export_ready', False):
    raise RuntimeError('先に「ビューアー用に変換」を実行してください。再学習は不要です。')
if not globals().get('MJSWAN_AVAILABLE', False):
    raise RuntimeError('mjswan のインストールに失敗しています。セットアップの出力を確認してください。')

from scripts.colab_client import build_walk

app = build_walk(
    walk_onnx_path='/content/RoboQuest2026/webapp/models/walk_policy_normalized.onnx',
    output_dir='/tmp/rq_walk_dist',
)
from scripts.launch_colab_viewer import launch_colab_viewer
if 'walk_viewer_server' in globals():
    walk_viewer_server.shutdown()
    walk_viewer_server.server_close()
walk_viewer_server = launch_colab_viewer('/tmp/rq_walk_dist', height=620)


🔧 Walk ビューアーをビルド中... (walk_policy_normalized.onnx)
Building the mjswan application...
Creating Node.js 25.5.0 environment in /content/RoboQuest2026/.venv-colab/lib/python3.13/site-packages/mjswan/template/.nodeenv
 * Install prebuilt node (25.5.0) ..... done.
Installing npm dependencies (npm install)...

added 275 packages, and audited 276 packages in 27s

54 packages are looking for funding
  run `npm fund` for details

11 vulnerabilities (6 moderate, 5 high)

To address all issues, run:
  npm audit fix

Run `npm audit` for details.
Running npm script: build:spa

> mjswan@0.8.2 build:spa
> tsc && vite build

Both esbuild and oxc options were set. oxc options will be used and esbuild options will be ignored. The following esbuild options were set: `{ drop: [ 'console', 'debugger' ] }`
vite v8.0.16 building client environment for production...

transforming...✓ 6976 modules transformed.
rendering chunks...
computing gzip size...
dist/assets/__vite-browser-external-DxA7VlKZ.js           

✅ 表示サーバー起動・Colab接続確認済み（3D描画完了の確認は画面で行ってください）


## 本番の鬼ごっこ：四脚の鬼1体対逃げ1体
5m×5m、最大60秒。鬼もGo2で、関節制御により歩いて追跡します。通常の目標速度は0.3m/sです。
2体の衝突用形状が接触した時点で捕獲します。逃げの胴体高さ0.15m未満が0.5秒継続、または場外で終了します。鬼の転倒では計時を止めません。
東西南北の4開始条件×5つの固定seedで20試合を行い、平均生存時間で順位を決めます。同点は目標速度0.4m/s、次に0.5m/sの追加20試合で比較し、なお同点なら同順位です。
最初の前進入門モデルの後、本番用の前後進・旋回・横移動を学び、四脚の鬼から逃げるモデルを学習します。
学習・事前評価・本番採点は同じ `competition.arena.Arena` を使用します。学習報酬を変えても公式スコアは変わりません。

本番用の低レベル観測には、歩行周期のsin/cosを公開入力として追加します。停止姿勢から確実に脚運びを開始できるよう、位相も入力して学習します。入門の45次元モデルは提出用の47次元モデルとは別です。


In [13]:
#@title 🐕 GitHubの学習済み四脚の鬼を読み込み・追跡確認（本番と同じモデル）
# GitHub同梱の学習済み鬼を読み込み、SHA-256を検査。本番も同じファイルを使用。
oni_report = prepare_oni(SAVE_DIR)
print('四脚の鬼：東西南北からの追跡・接触による捕獲に合格')
from IPython.display import Video, display
oni_video = record_oni_demo(SAVE_DIR)
display(Video(oni_video, embed=True, html_attributes='controls autoplay muted loop'))


四脚の鬼 east: 6.39s、captured、最低胴体高 0.277m
四脚の鬼 north: 7.04s、captured、最低胴体高 0.277m
四脚の鬼 west: 6.72s、captured、最低胴体高 0.277m
四脚の鬼 south: 7.41s、captured、最低胴体高 0.277m
四脚の鬼：東西南北からの追跡・接触による捕獲に合格


In [14]:
#@title 🦵 本番用の歩行を学習する（前後進・旋回・横移動・停止）
# 最初の前進入門モデルとは別に、本番の固定機体条件で未学習から学習します。
competition_walk_report = train_competition_walk(SAVE_DIR, seed=seed)
if not competition_walk_report['passed']:
    raise RuntimeError('本番用歩行の検証に不合格です。次へ進めません。')


本番用の歩行を未学習モデルから学習します（前後進・旋回・横移動・停止）。
歩行見本 10000/30000
歩行見本 20000/30000
見本学習 1000/4000: MSE 0.000043
見本学習 2000/4000: MSE 0.000018
見本学習 3000/4000: MSE 0.000011
見本学習 4000/4000: MSE 0.000009
本番用の再発進学習 1/3 完了
本番用の再発進学習 2/3 完了
本番用の再発進学習 3/3 完了
歩行PPO 10000ステップ
本番歩行PPO: 2000 ステップ完了
本番歩行PPO: 4000 ステップ完了
本番歩行PPO: 6000 ステップ完了
本番歩行PPO: 8000 ステップ完了
本番歩行PPO: 10000 ステップ完了
本番歩行PPO: 12000 ステップ完了
Loading PPO model: /content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4/competition_walk/walk_model
  VecNormalize stats embedded from /content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4/competition_walk/walk_model_vecnorm.pkl
Exported ONNX: /content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4/competition_walk/walk.onnx
正規化込みONNX照合: 最大誤差 3.5762787e-07
歩行検証 {'command': (0.0, 0.0, 0.0), 'passed': True, 'fell': False, 'distance_m': 0.04149926370802815, 'yaw_radians': 0.012387809493788409}
歩行検証 {'command': (0.4, 0.0, 0.0), 'passed': True, 'fell': False, 'distance_m': 7.563469384462922, 'yaw_radians': -0.3302021

## 3. 逃げ方の報酬を設計する
逃げモデルは5Hzで3個の速度指令を出し、歩行モデルは50Hzで12関節を制御します。観測は新仕様の19次元です。

| 範囲 | 内容・単位 |
|---|---|
| `[0:3]` | 自機基準の鬼の相対位置 x/y、水平距離（m） |
| `[3:7]` | 自機の世界位置 x/y（m）、向きのcos/sin |
| `[7:10]` | 自機基準の速度 x/y/z（m/s） |
| `[10:14]` | 東・北・西・南の壁までの距離（m） |
| `[14]` | 残り時間÷60秒 |
| `[15:17]` | 自機基準の鬼との相対速度 x/y（m/s） |
| `[17:19]` | 鬼の胴体高さ（m）、直立度 |

出力は `[-1,1]` の3値を、それぞれ前後0.5m/s、左右0.3m/s、旋回1rad/sの指令へ変換します。学習報酬は変更できますが、公式の採点は平均生存時間のみです。


In [15]:
#@title 🏃 逃げ学習の報酬設定（公式の鬼・採点条件は固定）
survival_weight = 1.0 #@param {type:"slider", min:0.0, max:3.0, step:0.1}
distance_weight = 0.1 #@param {type:"slider", min:0.0, max:1.0, step:0.1}
tag_penalty = 20.0 #@param {type:"slider", min:0.0, max:100.0, step:5.0}
flee_timesteps = 10000 #@param {type:"integer"}
print('鬼の通常目標速度は0.3m/s。報酬は学習用で、順位は生存時間だけで決まります。')


鬼の通常目標速度は0.3m/s。報酬は学習用で、順位は生存時間だけで決まります。


In [16]:
#@title 👹 逃げ方を学習して保存（四脚の鬼・本番の1対1環境）
train_competition_flee(SAVE_DIR, seed=seed, ppo_steps=flee_timesteps,
    reward={'survival': survival_weight, 'distance': distance_weight,
            'tag_penalty': tag_penalty, 'fall_penalty': 20.0})


実際の四脚の鬼から逃げる見本を収集します。
逃げ見本 3000/12000
逃げ見本 6000/12000
逃げ見本 9000/12000
見本学習 1000/2000: MSE 0.001455
見本学習 2000/2000: MSE 0.001086
逃げPPO 10000ステップ
逃げPPO: 2000 ステップ完了
逃げPPO: 4000 ステップ完了
逃げPPO: 6000 ステップ完了
逃げPPO: 8000 ステップ完了
逃げPPO: 10000 ステップ完了
Loading PPO model: /content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4/flee_model
  VecNormalize stats embedded from /content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4/flee_model_vecnorm.pkl
Exported ONNX: /content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4/flee.onnx
正規化込みONNX照合: 最大誤差 3.5762787e-07


{'version': 'rq2026-1v1-v1',
 'trained': True,
 'export_error': 3.5762786865234375e-07}

## 4. 同じ条件で結果を比較する
学習と評価で観測の尺度を揃える必要があります。`VecNormalize` は学習中に平均・分散を記録し、観測を正規化します。ZIPだけではこの情報が足りないため、PKLも一緒に保存します。
評価では保存した統計を読み込み、`training=False` で更新を止め、`norm_reward=False` で報酬の正規化を止めます。歩行・逃げの両方にそれぞれの観測統計を使います。

下のセルは新仕様の公式20試合（4開始条件×5seed）を評価します。生存時間は物理刻み0.002秒で記録します。数値結果と、保存したモデルが実際に四脚の鬼から逃げる試合動画を確認できます。正規化は提出ONNXに埋め込まれ、別のPKLは本番推論で読みません。


In [17]:
#@title 📦 公式提出パッケージを作成・検査（新仕様のみ）
# 利用した外部モデル・コードがあれば、出典と利用範囲を sources に追記してください。
sources = ['RoboQuest2026 教材の見本学習＋PPOコード']
submission_path = package_competition(SAVE_DIR, team=team_name, sources=sources)
print('検査済み提出ファイル:', submission_path)
print('提出期限まで再提出可能です。最後に検査を通過した提出物を採用します。')


検査済み提出ファイル: /content/drive/MyDrive/RoboQuest2026/隠居したいンゴ/shiken_4/submission.zip
提出期限まで再提出可能です。最後に検査を通過した提出物を採用します。


In [18]:
#@title 📊 保存した逃げAIを評価（公式20試合・再学習不要）
competition_report = evaluate_competition(SAVE_DIR, video=True)
print(f"平均生存時間: {competition_report['mean_survival_seconds']:.3f} 秒")
print(f"60秒完走: {competition_report['completions']}/20")
print('鬼1体対逃げ1体。東西南北×5 seed。距離ボーナスはありません。')
for row in competition_report['episodes']:
    print(row)


100/east: 18.804s captured
100/north: 16.740s captured
100/west: 14.804s captured
100/south: 6.816s captured
101/east: 18.842s captured
101/north: 14.722s captured
101/west: 14.770s captured
101/south: 6.768s captured
102/east: 18.808s captured
102/north: 15.018s captured
102/west: 14.530s captured
102/south: 6.778s captured
103/east: 18.974s captured
103/north: 14.994s captured
103/west: 15.414s captured
103/south: 6.938s captured
104/east: 18.824s captured
104/north: 15.020s captured
104/west: 14.758s captured
104/south: 6.952s captured
Results: /tmp/rq_match_zq82qio1/result.json
平均生存時間: 13.964 秒
60秒完走: 0/20
鬼1体対逃げ1体。東西南北×5 seed。距離ボーナスはありません。
{'seed': 100, 'spawn': 'east', 'oni_speed': 0.3, 'survived_seconds': 18.804000000000457, 'reason': 'captured', 'rule_version': 'rq2026-1v1-v1', 'rule_sha256': '87c82039c3b78cbd263aebd744b6176e35543da6d81522600d770bea3bb4bd45', 'oni_sha256': 'bdba05f0ee4778e56236904270b007ee4d0d446ce5b0721766e5a7109ef81977'}
{'seed': 100, 'spawn': 'north', 'oni_s

In [19]:
#@title 🎮 鬼ごっこビューアー（四脚の鬼と学習済み逃げAIの実試合動画）
#@markdown 公式評価の最初の試合を表示します。手動操作ではなく、保存した逃げAIの推論で動きます。
from IPython.display import Video, display
match_video = SAVE_DIR / 'competition_match.mp4'
if not match_video.is_file():
    raise RuntimeError('先に「保存した逃げAIを評価」を実行してください。')
display(Video(str(match_video), embed=True, html_attributes='controls autoplay muted loop'))


### 実験してみよう
1. `baseline` の結果を保存する。
2. 別の実験名にし、`orientation_weight=-2.0` で歩行から学習する。
3. 同じ速度指令で姿勢と追従を比較し、同じ評価seedで逃げ成績を比較する。
4. 「仮説・変更した値・結果・考察」を記録する。

逃げ報酬だけを比較する場合は、新しい実験フォルダに基準の `competition_walk` フォルダ（本番用モデルと合格記録） をコピーし、歩行学習を飛ばします。同じ歩行モデル・鬼の速度・評価seed・学習量を使うことで比較しやすくなります。

| 実験名 | 変更した値 | 平均生存秒 | 逃げ切り数 | 考察 |
|---|---|---|---|---|
| baseline | なし | 実行後に記入 | 実行後に記入 | |

さらに改造する場合は、`scripts/competition_workflow.py` の `train_competition_flee` が環境作成、PPO学習、保存を担当しています。環境内の報酬処理は `Go2WalkEnv` と `competition.gym_env.FleeEnv` にあります。まずこの教材の重み変更で比較方法を身につけてください。観測の追加はローカル実験として可能ですが、大会提出との互換性は下記を確認してください。


## 保存・提出と公式仕様
本番へ提出するファイルは `submission.zip` です。正規化込みONNXと `manifest.json` を含み、セルで新仕様の入力・出力、SHA256、推論を検査します。
学習用のモデルZIP・正規化統計は再学習用に保管します。大会への自動送信は行いません。
旧10次元・48次元モデルは互換性対象外です。階層型は高レベル19次元→3速度指令と低レベル47次元→12関節、直接制御型は66次元→12関節です。
鬼・自機・壁・残り時間の観測順序と単位、固定物理条件、提出形式は `competition/RULES.md` を参照してください。
PPO以外、見本学習、独自ネットワーク、報酬変更は可能です。提供観測からの特徴量変換は可能ですが、独自環境コードは本番で実行しません。
推論はCPU・外部通信なし、低レベル50Hz／高レベル5Hz。提出ZIP・展開後は各100MiB、メモリ2GiB、1スレッド、1回の推論100ms以内です。
公式20試合の結果は `competition_evaluation.json`、試合動画は `competition_match.mp4`、鬼の追跡試験は `oni_verification.json` に保存します。
公式環境の公開から提出締切まで最低7日を確保します。締切までは再提出可能で、最後に検査を通過した提出物を採用します。


In [ ]:
#@title 📂 保存済み歩行モデルを選んで表示（再学習不要）
#@markdown 表示する場合だけチェックしてください。通常の学習ではオフのままにします。
show_saved_walk = False #@param {type:"boolean"}
#@markdown RoboQuest2026 内の保存先を指定（例: チーム名/baseline、旧保存先は test）。
saved_team_name = "隠居したいンゴ/baseline" #@param {type:"string"}
#@markdown 実際のZIP名を指定（walkmodel.zipの場合は変更してください）。
saved_model_name = "walk_model.zip" #@param {type:"string"}
#@markdown 正規化ファイルが別名ならフルパスを指定。通常は空欄。
saved_vecnorm_path = "" #@param {type:"string"}

if show_saved_walk:
    from google.colab import drive
    drive.mount('/content/drive')

    from pathlib import Path
    root = Path('/content/drive/MyDrive/RoboQuest2026')
    folder = (root / saved_team_name).resolve()
    if not folder.is_relative_to(root.resolve()):
        raise ValueError('RoboQuest2026内のフォルダ名を指定してください')
    if not folder.is_dir():
        available = ', '.join(p.name for p in root.iterdir() if p.is_dir()) if root.exists() else '(Drive未接続)'
        raise FileNotFoundError(f'フォルダがありません。選べるフォルダ: {available}')
    model_path = folder / saved_model_name
    # Accept the common spelling without an underscore as well.
    if not model_path.exists() and saved_model_name == 'walk_model.zip':
        model_path = folder / 'walkmodel.zip'
    if not model_path.is_file():
        raise FileNotFoundError(f'ZIPがありません。候補: {[p.name for p in folder.glob("*.zip")]}')
    print(f'読み込むモデル: {model_path}')

    from scripts.colab_client import preview_saved_walk
    from scripts.launch_colab_viewer import launch_colab_viewer

    from scripts.colab_client import build_walk
    onnx_path, has_stats = preview_saved_walk(model_path, saved_vecnorm_path or None)
    app = build_walk(onnx_path, onnx_path.parent / 'viewer')
    if 'saved_walk_viewer_server' in globals():
        saved_walk_viewer_server.shutdown()
        saved_walk_viewer_server.server_close()
    saved_walk_viewer_server = launch_colab_viewer(onnx_path.parent / 'viewer', height=620)
    if not has_stats:
        from IPython.display import HTML, display
        display(HTML('<p style="color:#b45309"><b>参考プレビュー：正規化データなし。学習時の動作再現ではありません。</b></p>'))
else:
    print('保存済みモデルを表示するには show_saved_walk をオンにして、このセルを実行してください。')
